In [ ]:
# Cell 1 — Verify the SFT adapter is available
SFT_RUN_ID = "sft-YYYYMMDD-XXXX"  # from `alignforge registry list --kind sft`
PREF_HASH = "XXXXXXXXXXXX"  # from `alignforge data build` (preference dataset)
DRIVE_SYNC = "/content/drive/MyDrive/alignforge"

!alignforge registry list --kind sft   # confirm the SFT run is 'completed'

In [ ]:
# Cell 2 — DPO dry run (verify plan before spending GPU time)
!alignforge train dpo --config configs/train/dpo_qlora.yaml --model-config configs/model/qwen2_5_1_5b.yaml --sft-run {SFT_RUN_ID} --pref-hash {PREF_HASH} --dry-run

In [ ]:
# Cell 3 — Run DPO
!alignforge train dpo --config configs/train/dpo_qlora.yaml --model-config configs/model/qwen2_5_1_5b.yaml --sft-run {SFT_RUN_ID} --pref-hash {PREF_HASH}

In [ ]:
# Cell 4 — Inspect DPO metrics in the structured log
import json
import subprocess

result = subprocess.run(
    ["sh", "-c", "cat logs/alignforge.log | grep dpo_step_metrics"],
    capture_output=True,
    text=True,
)
lines = [json.loads(label) for label in result.stdout.splitlines() if label.strip()]
if lines:
    import pandas as pd

    df = pd.DataFrame(lines)[["step", "rewards_margin", "rewards_accuracy", "implicit_kl"]]
    print(df.tail(20).to_string(index=False))
    print(f"\nFinal mean KL: {df['implicit_kl'].mean():.4f}")

In [ ]:
# Cell 5 — Optionally run the beta sweep (costs 3x GPU time)
!alignforge train dpo-sweep --sft-run {SFT_RUN_ID} --pref-hash {PREF_HASH} --betas 0.05,0.1,0.3 --limit 500
# short runs for the sweep